# Deepfake Detection — Interactive Exploration

This notebook walks through the project step by step:
1. Visualise raw data (real vs fake)
2. Explore FFT frequency-domain representations
3. Load trained models and inspect predictions
4. Generate Grad-CAM heatmaps interactively
5. Compare all three models side-by-side

**Run from the project root** so that `src/` is importable.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Load a Small Sample of Images

In [ ]:
from src.dataset import get_dataloaders

# Load a tiny slice for exploration (200 train, 40 val, 40 test)
loaders = get_dataloaders(
    data_dir='../data',
    mode='rgb',
    batch_size=16,
    max_samples=200,
)

In [ ]:
from src.visualize import plot_sample_images

plot_sample_images(
    loaders['train'],
    save_path='sample_preview.png',
    num_per_class=4,
    title='Sample Real vs Fake Faces'
)
from IPython.display import Image
Image('sample_preview.png')

## 2. FFT Frequency-Domain Visualisation

Real faces (photographed) have smooth, isotropic frequency spectra.  
GAN-generated fakes often show grid-like patterns caused by transposed convolution upsampling.

In [ ]:
from src.fft_utils import save_fft_comparison, compute_fft_spectrum_gray

save_fft_comparison(loaders['train'], save_dir='.', num_samples=6)
Image('rgb_vs_fft_comparison.png')

In [ ]:
# Manual inspection: pick one real and one fake
batch_imgs, batch_labels = next(iter(loaders['train']))

real_idx = (batch_labels == 0).nonzero(as_tuple=True)[0][0]
fake_idx = (batch_labels == 1).nonzero(as_tuple=True)[0][0]

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, idx, name, color in [(0, real_idx, 'Real', 'green'),
                               (1, fake_idx, 'Fake', 'red')]:
    img = (batch_imgs[idx] * std + mean).clamp(0,1).permute(1,2,0).numpy()
    fft = compute_fft_spectrum_gray(batch_imgs[idx])
    axes[row][0].imshow(img)
    axes[row][0].set_title(f'RGB — {name}', color=color, fontsize=12)
    axes[row][0].axis('off')
    axes[row][1].imshow(fft, cmap='inferno')
    axes[row][1].set_title(f'FFT Spectrum — {name}', color=color, fontsize=12)
    axes[row][1].axis('off')

plt.suptitle('Frequency Artifacts in Fake Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Load Trained Models

In [ ]:
from src.models import build_model

def load_ckpt(exp, path):
    m = build_model(exp, backbone='resnet18', pretrained=False)
    m.load_state_dict(torch.load(path, map_location=device))
    m.to(device).eval()
    return m

rgb_model    = load_ckpt('rgb',    '../results/models/rgb_model.pth')
fft_model    = load_ckpt('fft',    '../results/models/fft_model.pth')
hybrid_model = load_ckpt('hybrid', '../results/models/hybrid_model.pth')
print('All three models loaded.')

## 4. Quick Inference Demo

In [ ]:
class_names = ['Real', 'Fake']

# Load one hybrid batch to get both rgb and fft tensors
hybrid_loaders = get_dataloaders('../data', mode='hybrid', batch_size=8, max_samples=200)
rgb_b, fft_b, lbls = next(iter(hybrid_loaders['test']))

with torch.no_grad():
    p_rgb    = rgb_model(rgb_b.to(device)).argmax(1).cpu()
    p_fft    = fft_model(fft_b.to(device)).argmax(1).cpu()
    p_hybrid = hybrid_model(rgb_b.to(device), fft_b.to(device)).argmax(1).cpu()

print(f"{'True':<6} {'RGB':<8} {'FFT':<8} {'Hybrid':<8}")
print('-' * 34)
for i in range(min(8, len(lbls))):
    true_s   = class_names[lbls[i]]
    rgb_s    = class_names[p_rgb[i]]
    fft_s    = class_names[p_fft[i]]
    hybrid_s = class_names[p_hybrid[i]]
    print(f"{true_s:<6} {rgb_s:<8} {fft_s:<8} {hybrid_s:<8}")

## 5. Grad-CAM Walkthrough

In [ ]:
from src.gradcam import GradCAM, overlay_heatmap

# Pick a single test image
test_rgb, test_lbl = next(iter(loaders['test']))
img  = test_rgb[0:1].to(device)

# Attach Grad-CAM to the RGB model's last conv layer
cam_obj  = GradCAM(rgb_model, rgb_model.features.layer4[-1].conv2)
heatmap, pred = cam_obj.generate(img)
cam_obj.remove_hooks()

overlay  = overlay_heatmap(test_rgb[0].cpu(), heatmap)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
img_vis = (test_rgb[0] * std + mean).clamp(0,1).permute(1,2,0).numpy()

axes[0].imshow(img_vis)
axes[0].set_title(f'Input — True: {class_names[test_lbl[0]]}', fontsize=10)
axes[0].axis('off')

axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title(f'Grad-CAM — Pred: {class_names[pred]}', fontsize=10)
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('Overlay', fontsize=10)
axes[2].axis('off')

plt.suptitle('Grad-CAM Explanation (RGB Model)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Compare All Model Metrics

In [ ]:
import json, pandas as pd

try:
    with open('../results/metrics/all_metrics_summary.json') as f:
        summary = json.load(f)
    df = pd.DataFrame(summary).T.round(4)
    df.index.name = 'Model'
    display(df.style.highlight_max(axis=0, color='#c8e6c9'))
except FileNotFoundError:
    print('Run evaluate_all.py first to generate the summary.')